# SFT LoRA Lesson

This notebook walks through a small supervised fine-tuning example using SFT and LoRA on a simple arithmetic task.

## Imports

Import the standard library, PyTorch, dataset tools, Transformers components, and the TRL and PEFT classes used throughout the lesson.

In [1]:
from typing import Any
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, TaskType
from rewards import compute_reward
from mlflow_tracking import (
    initialize_tracking,
    log_eval_result,
    log_history,
    log_json_artifact,
    log_metrics,
    log_params,
    start_child_run,
    start_parent_run,
)


/Users/jim/Desktop/genai/rft-learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path
import shutil

for directory in ["sft-arithmetic-lora-demo", "sft-arithmetic-lora-adapter"]:
    shutil.rmtree(Path(directory), ignore_errors=True)

print("Deleted any existing SFT output directories.")


Deleted any existing SFT output directories.


## Constants

Define the dataset bounds and the pretrained instruction model that will be evaluated and then fine-tuned.

In [3]:
MAX_A = 21
MAX_B = 11

model_name = "Qwen/Qwen2.5-0.5B-Instruct"


## Supervised Completion Builder

Build the canonical gold response that SFT will learn from. Unlike RFT, this lesson uses known target outputs instead of scalar scoring signals.

In [4]:
def make_completion(a: int, b: int) -> str:
    """Build the canonical supervised completion for one arithmetic example.

    Args:
        a: The first integer in the addition problem.
        b: The second integer in the addition problem.

    Returns:
        str: A gold completion in the required think-and-answer format.
    """
    total = a + b
    return f"<think>{a} + {b}</think><answer>{total}</answer>"


## Dataset Builder

Create a helper function that generates arithmetic prompts, expected answers for evaluation, and supervised prompt-completion pairs for SFT training.

In [5]:
def make_dataset() -> Dataset:
    """Build a small arithmetic dataset with prompt, completion, and answer fields.

    Args:
        None.

    Returns:
        Dataset: A Hugging Face dataset containing prompt-completion pairs and evaluation answers.
    """
    rows = []

    for a in range(1, MAX_A):
        for b in range(1, MAX_B):
            prompt_text = (
                f"What is {a} + {b}? Respond exactly as <think>{a} + {b}</think><answer>...</answer>"
            )
            total = a + b
            rows.append({
                "prompt_text": prompt_text,
                "prompt": [{"role": "user", "content": prompt_text}],
                "completion": [{"role": "assistant", "content": make_completion(a, b)}],
                "answer": str(total),
            })

    return Dataset.from_list(rows)


## Dataset Split

Build the dataset and split it into training and test subsets so we can compare behavior before and after fine-tuning.

In [6]:
dataset = make_dataset()
split = dataset.train_test_split(test_size=0.25, seed=42)

train_dataset = split["train"]
test_dataset = split["test"]

# show the first 5 examples from the training dataset
print("First 5 examples from the training dataset:")
for i in range(5):
    print(train_dataset[i])

# show the first 5 examples from the test dataset
print("First 5 examples from the test dataset:")
for i in range(5):
    print(test_dataset[i])


First 5 examples from the training dataset:
{'prompt_text': 'What is 9 + 3? Respond exactly as <think>9 + 3</think><answer>...</answer>', 'prompt': [{'role': 'user', 'content': 'What is 9 + 3? Respond exactly as <think>9 + 3</think><answer>...</answer>'}], 'completion': [{'role': 'assistant', 'content': '<think>9 + 3</think><answer>12</answer>'}], 'answer': '12'}
{'prompt_text': 'What is 10 + 9? Respond exactly as <think>10 + 9</think><answer>...</answer>', 'prompt': [{'role': 'user', 'content': 'What is 10 + 9? Respond exactly as <think>10 + 9</think><answer>...</answer>'}], 'completion': [{'role': 'assistant', 'content': '<think>10 + 9</think><answer>19</answer>'}], 'answer': '19'}
{'prompt_text': 'What is 9 + 10? Respond exactly as <think>9 + 10</think><answer>...</answer>', 'prompt': [{'role': 'user', 'content': 'What is 9 + 10? Respond exactly as <think>9 + 10</think><answer>...</answer>'}], 'completion': [{'role': 'assistant', 'content': '<think>9 + 10</think><answer>19</answer>'

## Shared Reward Module

Use the imported `compute_reward` helper from `rewards.py` for answer parsing, format validation, think-content validation, correctness checks, and combined scoring.

## Response Generation Helper

Define a helper that formats a prompt as a chat conversation, runs generation, and decodes only the new tokens.

In [7]:
def generate_response(model: Any, tokenizer: Any, prompt: str) -> str:
    """Generate a deterministic response for a single user prompt.

    Args:
        model: The causal language model used for generation.
        tokenizer: The tokenizer used to format and decode the prompt.
        prompt: The user prompt to send to the model.

    Returns:
        str: The decoded generated response text.
    """
    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


## Evaluation Helper

Define the evaluation routine that generates predictions across the test set and prints accuracy, format compliance, think compliance, and sample outputs.

In [8]:
def evaluate_model(model: Any, tokenizer: Any, eval_dataset: Dataset, label: str) -> dict[str, Any]:
    """Run evaluation on a dataset, print summary metrics, and return structured results."""
    model.eval()

    total = len(eval_dataset)
    correct = 0
    formatted = 0
    think_valid = 0
    total_score = 0.0
    total_format_reward = 0.0
    total_correctness_reward = 0.0
    total_think_reward = 0.0

    samples = []
    rows = []

    for row in eval_dataset:
        prompt = row["prompt_text"]
        expected = row["answer"]

        text = generate_response(model, tokenizer, prompt)
        reward = compute_reward(text, expected, prompt)
        score = reward.total_reward

        correct += int(reward.is_correct)
        formatted += int(reward.is_formatted)
        think_valid += int(reward.is_think_valid)
        total_score += score
        total_format_reward += reward.format_reward
        total_correctness_reward += reward.correctness_reward
        total_think_reward += reward.think_reward

        eval_row = {
            "prompt": prompt,
            "expected": expected,
            "generated": text,
            "predicted": reward.predicted_answer,
            "is_correct": reward.is_correct,
            "is_formatted": reward.is_formatted,
            "is_think_valid": reward.is_think_valid,
            "format_reward": reward.format_reward,
            "correctness_reward": reward.correctness_reward,
            "think_reward": reward.think_reward,
            "score": score,
        }
        rows.append(eval_row)

        if len(samples) < 5:
            samples.append({
                "prompt": prompt,
                "expected": expected,
                "generated": text,
                "predicted": reward.predicted_answer,
                "think_reward": reward.think_reward,
                "score": score,
            })

    metrics = {
        "prompt_count": total,
        "completion_count": total,
        "accuracy": correct / total,
        "format_compliance": formatted / total,
        "think_compliance": think_valid / total,
        "avg_reward": total_score / total,
        "avg_score": total_score / total,
        "avg_format_reward": total_format_reward / total,
        "avg_correctness_reward": total_correctness_reward / total,
        "avg_think_reward": total_think_reward / total,
    }
    result = {
        "label": label,
        "metrics": metrics,
        "settings": {
            "num_generations": 1,
            "do_sample": False,
        },
        "samples": samples,
        "rows": rows,
    }

    print(f"\n=== {label} ===")
    print(f"Answer accuracy:   {correct}/{total} = {metrics['accuracy']:.2%}")
    print(f"Format compliance: {formatted}/{total} = {metrics['format_compliance']:.2%}")
    print(f"Think compliance:  {think_valid}/{total} = {metrics['think_compliance']:.2%}")
    print(f"Average score:     {metrics['avg_score']:.3f}")

    print("\nSample generations:")
    for ex in samples:
        print("-" * 60)
        print("Prompt:   ", ex["prompt"])
        print("Expected: ", ex["expected"])
        print("Generated:", ex["generated"])
        print("Predicted:", ex["predicted"])
        print("Think reward:", ex["think_reward"])
        print("Score:    ", ex["score"])

    return result


## Tokenizer Setup

Load the tokenizer for the base instruction model so prompts can be formatted and outputs decoded.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_name)


## Base Model Setup

Load the pretrained causal language model and choose a practical dtype depending on whether CUDA is available.

In [10]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:01<00:00, 264.17it/s]


## Baseline Evaluation

Measure how the base model performs on the held-out arithmetic examples before supervised fine-tuning.

In [11]:
tracking_setup = initialize_tracking(experiment_name="rft-learning")
parent_run_manager = start_parent_run(
    notebook_name="sft-lora-lesson.ipynb",
    notebook_type="sft",
    tags={"model_name": model_name},
)
parent_run = parent_run_manager.__enter__()

log_params(
    {
        "model_name": model_name,
        "notebook_type": "sft",
        "dataset": {
            "train_size": len(train_dataset),
            "test_size": len(test_dataset),
            "split_seed": 42,
        },
    },
    prefix="run",
)
log_json_artifact(
    "sft_run_context.json",
    {
        "tracking": {
            "tracking_uri": tracking_setup.tracking_uri,
            "experiment_name": tracking_setup.experiment_name,
            "tracking_path": str(tracking_setup.tracking_path),
        },
        "dataset": {
            "train_size": len(train_dataset),
            "test_size": len(test_dataset),
            "split_seed": 42,
        },
    },
    artifact_path="run_context",
)

baseline_eval_result = evaluate_model(
    base_model,
    tokenizer,
    test_dataset,
    label="Before SFT + LoRA"
)

with start_child_run("baseline_eval"):
    log_eval_result("baseline_eval", baseline_eval_result)

log_metrics(baseline_eval_result["metrics"], prefix="baseline")


2026/06/13 22:13:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/13 22:13:11 INFO mlflow.store.db.utils: Updating database tables



=== Before SFT + LoRA ===
Answer accuracy:   0/50 = 0.00%
Format compliance: 0/50 = 0.00%
Think compliance:  0/50 = 0.00%
Average score:     0.000

Sample generations:
------------------------------------------------------------
Prompt:    What is 16 + 3? Respond exactly as <think>16 + 3</think><answer>...</answer>
Expected:  19
Generated: The result of 16 + 3 is 19.
Predicted: 
Think reward: 0.0
Score:     0.0
------------------------------------------------------------
Prompt:    What is 3 + 7? Respond exactly as <think>3 + 7</think><answer>...</answer>
Expected:  10
Generated: 3 + 7 = 10
Predicted: 
Think reward: 0.0
Score:     0.0
------------------------------------------------------------
Prompt:    What is 12 + 6? Respond exactly as <think>12 + 6</think><answer>...</answer>
Expected:  18
Generated: The result of 12 + 6 is 18.
Predicted: 
Think reward: 0.0
Score:     0.0
------------------------------------------------------------
Prompt:    What is 5 + 7? Respond exactly as <th

{'baseline.prompt_count': 50.0,
 'baseline.completion_count': 50.0,
 'baseline.accuracy': 0.0,
 'baseline.format_compliance': 0.0,
 'baseline.think_compliance': 0.0,
 'baseline.avg_reward': 0.0,
 'baseline.avg_score': 0.0,
 'baseline.avg_format_reward': 0.0,
 'baseline.avg_correctness_reward': 0.0,
 'baseline.avg_think_reward': 0.0}

## LoRA Configuration

Configure the LoRA adapter modules and hyperparameters that will be attached during supervised fine-tuning.

In [12]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


## SFT Training Arguments

Set the supervised fine-tuning hyperparameters, including output location, batch sizes, sequence length, and completion-only loss.

In [13]:
training_args = SFTConfig(
    output_dir="sft-arithmetic-lora-demo",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_length=192,
    completion_only_loss=True,
    bf16=torch.cuda.is_available(),
    use_cpu=not torch.cuda.is_available(),
    num_train_epochs=1,
    logging_steps=10,
    learning_rate=5e-5,
)


## Trainer Construction

Create the SFT trainer by connecting the base model, tokenizer, training dataset, and LoRA configuration.

In [14]:
trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

log_params(
    {
        "lora": lora_config,
        "training": training_args,
    },
)
log_json_artifact(
    "sft_training_config.json",
    {
        "base_model_name": model_name,
        "lora_config": lora_config,
        "training_args": training_args,
    },
    artifact_path="configs",
)


Tokenizing train dataset: 100%|██████████| 150/150 [00:00<00:00, 3974.76 examples/s]


## Training And Saving

Run supervised fine-tuning and save the resulting adapter weights so they can be reused later.

In [15]:
with start_child_run("training"):
    train_result = trainer.train()
    adapter_output_dir = "sft-arithmetic-lora-adapter"
    trainer.save_model(adapter_output_dir)
    training_summary = {
        "output_dir": training_args.output_dir,
        "adapter_output_dir": adapter_output_dir,
        "train_result_metrics": train_result.metrics,
    }
    log_params({"output_dir": training_args.output_dir, "adapter_output_dir": adapter_output_dir}, prefix="training")
    log_metrics(train_result.metrics)
    log_json_artifact("sft_training_summary.json", training_summary, artifact_path="training")
    log_history(trainer.state.log_history, "sft_trainer_log_history")

log_metrics(train_result.metrics, prefix="training")


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.191947


{'training.train_runtime': 53.7032,
 'training.train_samples_per_second': 2.793,
 'training.train_steps_per_second': 0.354,
 'training.total_flos': 25336453097472.0,
 'training.train_loss': 0.10119424158658244,
 'training.entropy': 0.0034871040909950223,
 'training.num_tokens': 11367.0,
 'training.mean_token_accuracy': 1.0,
 'training.epoch': 1.0}

## Trained Model Reference

Grab the trainer's model handle so the post-training evaluation step can use the updated weights.

In [16]:
trained_model = trainer.model


## Post-Training Evaluation

Evaluate the trained model on the same held-out dataset to compare behavior after supervised fine-tuning.

In [17]:
trained_model = trainer.model

fine_tuned_eval_result = evaluate_model(
    trained_model,
    tokenizer,
    test_dataset,
    label="After SFT + LoRA"
)

with start_child_run("fine_tuned_eval"):
    log_eval_result("fine_tuned_eval", fine_tuned_eval_result)

log_metrics(fine_tuned_eval_result["metrics"], prefix="fine_tuned")
parent_run_manager.__exit__(None, None, None)



=== After SFT + LoRA ===
Answer accuracy:   50/50 = 100.00%
Format compliance: 50/50 = 100.00%
Think compliance:  50/50 = 100.00%
Average score:     2.500

Sample generations:
------------------------------------------------------------
Prompt:    What is 16 + 3? Respond exactly as <think>16 + 3</think><answer>...</answer>
Expected:  19
Generated: <think>16 + 3</think><answer>19</answer>
Predicted: 19
Think reward: 1.0
Score:     2.5
------------------------------------------------------------
Prompt:    What is 3 + 7? Respond exactly as <think>3 + 7</think><answer>...</answer>
Expected:  10
Generated: <think>3 + 7</think><answer>10</answer>
Predicted: 10
Think reward: 1.0
Score:     2.5
------------------------------------------------------------
Prompt:    What is 12 + 6? Respond exactly as <think>12 + 6</think><answer>...</answer>
Expected:  18
Generated: <think>12 + 6</think><answer>18</answer>
Predicted: 18
Think reward: 1.0
Score:     2.5
----------------------------------------

False